In [13]:
# This script trains the DeepHit model on the provided dataset and 
# evaluates its performance using the C-index at a specified time horizon.
# It also includes a permutation feature importance analysis to identify
# which features are most influential for the model's predictions at that horizon.
import os
import random
import numpy as np
import tensorflow as tf
from trainDeepHit import *
from sklearn.model_selection import KFold
from utils_eval import c_index
import pandas as pd
import ImportDatabase as impt
from ClassDeepHit import DeepHitPlus
from sklearn.model_selection import train_test_split
from dotenv import load_dotenv
from pathlib import Path

load_dotenv()

True

In [9]:
SEED = 321

num_Event = 2
eval_time = 12
CV_ITERATION = 5

def setSeed(seed=SEED):
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ["TF_DETERMINISTIC_OPS"] = "1"   # deterministic TF ops
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

setSeed(SEED)

In [14]:
# ==================================================
# Load data
# ==================================================
data_dir = Path(os.getenv("DATA_DIR"))
path = data_dir / "Region_Sthlm_Gotland_model_data_binary.csv"

x_dim, (data, time, label), (mask1, mask2) = impt.import_dataset(path,time_scale=1.2)

# Normalize age and log-transform certain features
data[:, 0] = (data[:, 0] - data[:, 0].mean()) / data[:, 0].std()

for j in range(data.shape[1] - 3, data.shape[1]): 
    data[:, j] = np.log1p(data[:, j])
    data[:, j] = (data[:, j] - data[:, j].mean()) / data[:, j].std()


In [4]:
# --------------------------------------------------
# Ensure TensorFlow-compatible dtypes
# --------------------------------------------------
data   = data.astype(np.float32)
mask1  = mask1.astype(np.float32)
mask2  = mask2.astype(np.float32)
time   = time.astype(np.float32)
label  = label.astype(np.int32)   


# ==================================================
# Train / Validation split
# ==================================================
kf = KFold(n_splits=CV_ITERATION, shuffle=True, random_state=SEED)
splits = list(kf.split(data))
for i in range(CV_ITERATION):
    tf.keras.backend.clear_session()
    setSeed(SEED)
    train_index, val_index = splits[i]

    tr_data     = data[train_index]
    tr_time     = time[train_index]
    tr_label    = label[train_index]
    tr_mask1    = mask1[train_index]
    tr_mask2    = mask2[train_index]

    va_data     = data[val_index]
    va_time     = time[val_index]
    va_label    = label[val_index]
    va_mask1    = mask1[val_index]
    va_mask2    = mask2[val_index]
    # ==================================================
    # Model definition
    # ==================================================
    num_bins = mask2.shape[1]   # T (time bins)

    model = DeepHitPlus(
        num_event=num_Event,
        num_bins=num_bins,
        shared_layers=(30, 15, 5),
        MR_layers=[5, 10],
        CR_layers=(5, 10),
        dropout=0.25   
    )

    # Build model (optional sanity check)
    pmf = model(tr_data[:2], training=False)
    print("PMF shape:", pmf.shape)   # (B, K, T)

    model = TrainDeepHit(model, tr_data, tr_mask1, tr_mask2, tr_time, tr_label,
                    va_data, va_mask1, va_mask2, va_time, va_label, eval_time = [12], 
                    test_size = 0.2, max_epochs = 100, batch_size = 128, learning_rate = 1e-3, alpha=1.0, beta=1.0)

    print("\n===== TRAINING FINISHED =====\n")


    def compute_cindex_at_time(model, data, time, label, eval_horizon):
        """
        Compute event-specific C-index at a given time horizon.
        """
        pred = model(data, training=False).numpy()  # (N, K, T)

        if eval_horizon >= pred.shape[2]:
            raise ValueError("eval_horizon exceeds num_Category")

        # Risk up to horizon
        risk = np.sum(pred[:, :, :(eval_horizon + 1)], axis=2)  # (N, K)

        cidx = np.zeros(num_Event)
        for k in range(num_Event):
            cidx[k] = c_index(
                risk[:, k],
                time,
                (label[:, 0] == k + 1).astype(int),
                eval_horizon
            )
        return cidx

    eval_horizon = 12  # same horizon you report in results

    baseline_cindex = compute_cindex_at_time(
        model,
        va_data,
        va_time,
        va_label,
        eval_horizon
    )

    print("Baseline C-index:", baseline_cindex)

PMF shape: (2, 2, 14)

===== START TRAINING =====

Epoch 000 | Train 4.4840 (LL 2.0990, Rank 2.3850) | Val 44.7273
Epoch 001 | Train 3.8153 (LL 1.5944, Rank 2.2208) | Val 34.7802
Epoch 002 | Train 3.3327 (LL 1.2283, Rank 2.1044) | Val 36.8325
Epoch 003 | Train 3.0292 (LL 1.0091, Rank 2.0201) | Val 34.2603
Epoch 004 | Train 2.8292 (LL 0.8798, Rank 1.9493) | Val 32.7038
Epoch 005 | Train 2.6783 (LL 0.7896, Rank 1.8887) | Val 31.8670
Epoch 006 | Train 2.5574 (LL 0.7238, Rank 1.8337) | Val 31.0012
Epoch 007 | Train 2.4616 (LL 0.6743, Rank 1.7873) | Val 30.4121
Epoch 008 | Train 2.3788 (LL 0.6351, Rank 1.7437) | Val 30.1991
Epoch 009 | Train 2.3117 (LL 0.6030, Rank 1.7087) | Val 30.1041
Epoch 010 | Train 2.2510 (LL 0.5763, Rank 1.6747) | Val 29.3658
Epoch 011 | Train 2.1954 (LL 0.5540, Rank 1.6414) | Val 30.1375
Epoch 012 | Train 2.1466 (LL 0.5350, Rank 1.6116) | Val 29.6400
Epoch 013 | Train 2.1034 (LL 0.5188, Rank 1.5846) | Val 28.8807
Epoch 014 | Train 2.0622 (LL 0.5043, Rank 1.5579) | V

In [5]:
# Final traing of the model after cross-validation, with the best hyperparameters, on the entire dataset
(tr_data, va_data,
    tr_time, va_time,
    tr_label, va_label,
    tr_mask1, va_mask1,
    tr_mask2, va_mask2) = train_test_split(
    data, time, label, mask1, mask2,
    test_size=0.2,
    random_state=SEED
)

model = TrainDeepHit(model, tr_data, tr_mask1, tr_mask2, tr_time, tr_label,
                va_data, va_mask1, va_mask2, va_time, va_label, eval_time = [12], 
                test_size = 0.2, max_epochs = 100, batch_size = 128, learning_rate = 1e-3, alpha=1.0, beta=1.0)



===== START TRAINING =====

Epoch 000 | Train 1.5501 (LL 0.3054, Rank 1.2446) | Val 30.6794
Epoch 001 | Train 1.5327 (LL 0.2999, Rank 1.2328) | Val 30.4334
Epoch 002 | Train 1.5194 (LL 0.3007, Rank 1.2187) | Val 30.8711
Epoch 003 | Train 1.5051 (LL 0.3001, Rank 1.2050) | Val 30.8150

Best C-index: 0.890 (competing event: 0.938)



In [6]:
def permutation_feature_importance(
    model,
    x_val,
    time_val,
    label_val,
    eval_horizon,
    n_repeats=5
):
    """
    Permutation feature importance using drop in C-index.
    Returns array of shape (num_Event, num_features).
    """
    n_features = x_val.shape[1]
    baseline = compute_cindex_at_time(
        model, x_val, time_val, label_val, eval_horizon
    )
    
    importance = np.zeros((num_Event, n_features))

    for j in range(n_features):
        drops = []

        for _ in range(n_repeats):
            x_perm = x_val.copy()
            perm_idx = np.random.permutation(x_perm.shape[0])
            x_perm[:, j] = x_perm[perm_idx, j]

            cidx_perm = compute_cindex_at_time(
                model, x_perm, time_val, label_val, eval_horizon
            )

            drops.append(baseline - cidx_perm)

        importance[:, j] = np.mean(drops, axis=0)

        if j % 10 == 0:
            print(f"PFI done for feature {j}/{n_features}")

    return importance

pfi = permutation_feature_importance(
    model=model,
    x_val=va_data,
    time_val=va_time,
    label_val=va_label,
    eval_horizon=eval_horizon,
    n_repeats=3
)

print("PFI shape:", pfi.shape)  # (num_Event, num_features)

# Top 10 features per event
for k in range(num_Event):
    top_idx = np.argsort(-pfi[k])[:10]
    print(f"\nTop features for event {k+1}:")
    print(top_idx)
    print(pfi[k, top_idx])


PFI done for feature 0/88
PFI done for feature 10/88
PFI done for feature 20/88
PFI done for feature 30/88
PFI done for feature 40/88
PFI done for feature 50/88
PFI done for feature 60/88
PFI done for feature 70/88
PFI done for feature 80/88
PFI shape: (2, 88)

Top features for event 1:
[ 0  6  9 32  7  3 87 28 85 15]
[0.05067619 0.00998855 0.00796549 0.00567337 0.00561387 0.00537328
 0.00491408 0.00412374 0.00398792 0.00398792]

Top features for event 2:
[ 0 87  7  3  9 85  2 71 60 68]
[0.02782438 0.02136817 0.01840272 0.01716571 0.01260061 0.0094691
 0.00840154 0.00590378 0.00515141 0.00488367]


In [ ]:
    
path = np.array([
    data_dir / "Region_Uppsala_Örebro_model_data_binary.csv",
    data_dir / "Region_Väst_model_data_binary.csv",
    data_dir / "Region_Syd_model_data_binary.csv",
    data_dir / "Region_Sydöstra_model_data_binary.csv",
    data_dir / "Region_Norr_model_data_binary.csv"]
    )
Region_names = {
    0: "Region Uppsala_Örebro",
    1: "Region Väst",
    2: "Region Syd",
    3: "Region Sydöstra",
    4: "Region Norr"
}

for i in range(len(path)):

    x_dim, (data, time, label), (mask1, mask2) = impt.import_dataset(path[i],time_scale=1.2)

    num_Category = mask2.shape[1]   # T (time bins)
    eval_time = [12]
    data[:, 0] = (data[:, 0] - data[:, 0].mean()) / data[:, 0].std()

    for j in range(data.shape[1] - 3, data.shape[1]):
        data[:, j] = np.log1p(data[:, j])
        data[:, j] = (data[:, j] - data[:, j].mean()) / data[:, j].std()

    data   = data.astype(np.float32)
    mask1  = mask1.astype(np.float32)
    mask2  = mask2.astype(np.float32)
    time   = time.astype(np.float32)
    label  = label.astype(np.int32) 

    pred = model.call(data)
    resultfeat = np.zeros([num_Event, len(eval_time)])

    for t, t_time in enumerate(eval_time):
        eval_horizon = int(t_time)

        if eval_horizon >= num_Category:
            print('ERROR: evaluation horizon is out of range')
            resultfeat[:, t] = -1
        else:
            # calculate F(t | x, Y, t >= t_M) = \sum_{t_M <= \tau < t} P(\tau | x, Y, \tau > t_M)
            risk = np.sum(pred[:,:,:(eval_horizon+1)], axis=2) #risk score until eval_time

            # Calculate permutation importance for each eval horizon and event
            for k in range(num_Event):
                resultfeat[k, t] = c_index(risk[:,k], time, (label[:,0] == k+1).astype(int), eval_horizon) #-1 for no event (not comparable)
                
                #result.loc[k+1, feat, perm_iter, "Delta t = " + "{:03d}".format(t_time)] = result1[k,t] - resultfeat[k,t]
                # since we compare risk_scores, the true label is that event occurs before time horizon
                #result.loc[k+1, feat, perm_iter, "Delta t = " + "{:03d}".format(t_time)] = result1[k,t] - resultfeat[k,t]
                # since we compare risk_scores, the true label is that event occurs before time horizon
    print(
        f"{Region_names[i]:<25}\t"
        f"main risk: {resultfeat[0,0]:.3f}\t"
        f"competing risk: {resultfeat[1,0]:.3f}"
    )


Region Uppsala_Örebro   	main risk: 0.842	competing risk: 0.822
Region Väst             	main risk: 0.830	competing risk: 0.842
Region Syd               	main risk: 0.820	competing risk: 0.811
Region Sydöstra         	main risk: 0.830	competing risk: 0.840
Region Norr              	main risk: 0.805	competing risk: 0.813
